# Lesson 11: Recurrent Architectures and Time Series - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: sequence memory, lag features, LSTM/GRU intuition, and remaining useful life.

## Learning goals
1. Generate a degrading health-index sequence.
2. Build supervised sliding windows.
3. Train a simple sequence forecaster.
4. Estimate remaining useful life from the forecast.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(11)
t = np.arange(160)
health = 9.5 - 0.025*t - 0.00025*t**2
health[75:] -= 0.03*(t[75:]-75)
health += rng.normal(0, 0.12, size=t.shape)

window = 12
X, y = [], []
for i in range(window, len(health)):
    X.append(health[i-window:i])
    y.append(health[i])
X, y = np.array(X), np.array(y)
split = 120 - window

model = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32,), max_iter=3000, random_state=1))
model.fit(X[:split], y[:split])
pred = model.predict(X[split:])
print("Test RMSE:", mean_squared_error(y[split:], pred, squared=False))

history = list(health[:120])
future = []
for _ in range(60):
    x_next = np.array(history[-window:]).reshape(1, -1)
    y_next = model.predict(x_next)[0]
    history.append(y_next)
    future.append(y_next)

future_t = np.arange(120, 180)
cross = next((tt for tt, h in zip(future_t, future) if h < 2.5), None)
plt.figure(figsize=(8, 4))
plt.plot(t, health, label="observed")
plt.plot(future_t, future, "--", label="forecast")
plt.axhline(2.5, color="tab:red", ls="--", label="failure threshold")
if cross:
    plt.axvline(cross, color="tab:red", ls=":")
plt.xlabel("time")
plt.ylabel("health index")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("Predicted threshold crossing:", cross)
print("Sliding windows are a simple approximation to recurrent memory; LSTMs/GRUs learn the memory update directly.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?